In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import os
import subprocess
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd, remove_tick_and_label

In [ ]:
# Settings
seed = 42

SINGLE_DIR = "single_sample"
PATIENT_DIR = "patient"
ALL_DIR = "all"

In [ ]:
libd_dlpfc_dir = DATA_DIR / "LIBD_DLPFC"
libd_dlpfc_results = RESULT_DIR / "LIBD_DLPFC"

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
adata = ad.read_h5ad(libd_dlpfc_dir / "raw.h5ad")

del adata.obsm
del adata.uns
del adata.layers

adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy().astype(int)

adata.var = adata.var[["gene_name", "type", "gene_biotype"]]

adata.obs = adata.obs[["sample_id", "subject", "discard", "spatialLIBD"]].rename(
    columns={"spatialLIBD": "label"}
)

adata.X = adata.X.tocsr().astype("int32")

# Preprocess

In [ ]:
# whether to center data for scaling and PCA
center = True

## Single sample

In [ ]:
out_dir = libd_dlpfc_dir / SINGLE_DIR
out_dir.mkdir(exist_ok=True, parents=True)

for patient_sample, idx in adata.obs.groupby(
    ["subject", "sample_id"], observed=True
).indices.items():
    sample = "_".join(patient_sample)
    adata_sample = adata[idx, :].copy()
    adata_sample.uns["n_clusters"] = adata_sample.obs["label"].nunique()

    preprocess_genes(adata_sample, batch_key=None)
    preprocess_scaling(adata_sample, center=center, seed=seed)

    adata_sample.write(out_dir / f"{sample}.h5ad")

## Patient-wise

In [ ]:
out_dir = libd_dlpfc_dir / PATIENT_DIR
out_dir.mkdir(exist_ok=True, parents=True)

for patient, idx in adata.obs.groupby("subject", observed=True).indices.items():
    adata_patient = adata[idx, :].copy()
    adata_patient.uns["n_clusters"] = adata_patient.obs["label"].nunique()
    adata_patient.uns["batch_keys"] = ["sample_id"]
    adata_patient.obs_names_make_unique()

    preprocess_genes(adata_patient, batch_key=None)
    preprocess_scaling(adata_patient, center=center, seed=seed)

    adata_patient.write(out_dir / f"{patient}.h5ad")

## All

In [ ]:
out_dir = libd_dlpfc_dir / ALL_DIR
out_dir.mkdir(exist_ok=True, parents=True)

adata.obs_names_make_unique()
adata.uns["n_clusters"] = adata.obs["label"].nunique()
adata.uns["batch_keys"] = ["sample_id", "subject"]

preprocess_genes(adata, batch_key=adata.uns["batch_keys"][0])
preprocess_scaling(adata, center=center, seed=seed)

adata.write(out_dir / "LIBD_DLPFC.h5ad")

# Batch integration

In [ ]:
script_dir = Path("batch_integration")

## Single sample (i.e. no integration)

In [ ]:
conda_cmd = activate_env_cmd("no_batch_integration")

In [ ]:
file_dir = libd_dlpfc_dir / SINGLE_DIR
script_path = script_dir / "no_batch_integration.py"

for h5ad in file_dir.glob("*.h5ad"):
    name = h5ad.stem
    out_dir = libd_dlpfc_results / h5ad.parent.name
    out_dir.mkdir(exist_ok=True, parents=True)
    out_file = out_dir / f"{name}.npz"

    cmd = f"{script_path} {h5ad} {out_file}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

## Patient-wise / All

In [ ]:
methods = [
    "no_batch_integration",
    "bbknn",
    "harmony",
    "scvi",
    "scanorama",
]

In [ ]:
files = list((libd_dlpfc_dir / PATIENT_DIR).glob("*.h5ad")) + list(
    (libd_dlpfc_dir / ALL_DIR).glob("*.h5ad")
)

In [ ]:
for h5ad in files:
    name = h5ad.stem

    for method in methods:
        out_dir = libd_dlpfc_results / method / h5ad.parent.name
        out_dir.mkdir(exist_ok=True, parents=True)
        out_file = out_dir / f"{name}.npz"

        script_path = script_dir / f"{method}.py"

        conda_cmd = activate_env_cmd(method)
        cmd = f"{script_path} {h5ad} {out_file}"

        # because bbknn uses n neighbors per batch we decrease the number of neighbors
        # so its roughly equal to the rest
        if method == "bbknn":
            n = 5  # change based on within or across patients?
            cmd += f" --n_neighs {n}"
        elif method == "scvi":
            n_comp = 20
            cmd += f" --n_comp {n_comp}"

        subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
script_dir = Path("clustering")

In [ ]:
conda_cmd = activate_env_cmd("spatialleiden_multisample")
script_path = script_dir / "spatialleiden_multisample.py"

In [ ]:
layer_ratio = 0.8

## Single sample

In [ ]:
file_dir = libd_dlpfc_dir / SINGLE_DIR

for h5ad in file_dir.glob("*.h5ad"):
    name = h5ad.stem
    out_dir = libd_dlpfc_results / h5ad.parent.name
    neighbor_file = out_dir / f"{name}.npz"
    out_file = out_dir / f"{name}.parquet"

    cmd = f"{script_path} {h5ad} {neighbor_file} {out_file}"
    # cmd+=f" --n_cluster {n_cluster}"
    cmd += f" --weight {layer_ratio}"
    cmd += " --coord_type grid"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

## Patient-wise / All

In [ ]:
for file in libd_dlpfc_results.glob("*/*/*.npz"):
    name = file.stem
    batch = file.parent.name
    method = file.parents[1].name

    if method != "scanorama":
        continue

    h5ad = libd_dlpfc_dir / batch / f"{name}.h5ad"
    out_file = file.parent / f"{name}.parquet"

    cmd = f"{script_path} {h5ad} {file} {out_file}"
    # bbknn has different number of neighbors therefore different weight is required
    # increase spatial weight to counteract increase #neighbors in latent space
    if method != "bbknn":
        cmd += f" --weight {layer_ratio}"
    else:
        cmd += " --weight 1.2"
    cmd += " --coord_type grid"
    cmd += " --use_batch"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Evaluation

In [ ]:
scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

integration = {
    "no_batch_integration": "no integration",
    "bbknn": "BBKNN",
    "harmony": "Harmony",
    "scvi": "scVI",
    "scanorama": "Scanorama",
}

In [ ]:
fig, axs = plt.subplots(nrows=11, ncols=12, figsize=(20, 18))

# single sample
labels = []
for file in libd_dlpfc_results.glob("*/*.parquet"):
    patient, sample = file.stem.split("_")
    labels.append(pd.read_parquet(file).assign(patient=patient, sample=sample))
labels = pd.concat(labels).astype(
    {"patient": "category", "spatialleiden": "category", "sample": "category"}
)

i = 0
for p, p_df in labels.groupby("patient", observed=True):
    for s, df in p_df.groupby("sample", observed=True):
        ax = axs[0, i]
        sns.scatterplot(
            data=df,
            x="x",
            y="y",
            hue="spatialleiden",
            ax=ax,
            legend=False,
            **scatter_kwargs,
        )
        remove_tick_and_label(ax)
        ax.set(aspect=1, title=sample)
        if i == 0:
            ax.set(ylabel="single")
        i += 1

# patient-wise
labels = []
for file in libd_dlpfc_results.glob("*/patient/*.parquet"):
    tool = file.parents[1].name
    patient = file.stem
    labels.append(pd.read_parquet(file).assign(patient=patient, tool=tool))
labels = (
    pd.concat(labels)
    .rename(columns={"sample_id": "sample"})
    .astype({"patient": "category", "spatialleiden": "category", "sample": "category"})
)

i = 1
for t, t_df in labels.groupby("tool", observed=True):
    j = 0
    for p, p_df in t_df.groupby("patient", observed=True):
        for s, df in p_df.groupby("sample", observed=True):
            ax = axs[i, j]
            sns.scatterplot(
                data=df,
                x="x",
                y="y",
                hue="spatialleiden",
                ax=ax,
                legend=False,
                **scatter_kwargs,
            )
            remove_tick_and_label(ax)
            ax.set(aspect=1)
            if j == 0:
                ax.set(ylabel=t)
            j += 1
    i += 1

# all
labels = []
for file in libd_dlpfc_results.glob("*/all/*.parquet"):
    tool = file.parents[1].name
    labels.append(pd.read_parquet(file).assign(tool=tool))
labels = (
    pd.concat(labels)
    .rename(columns={"subject": "patient", "sample_id": "sample"})
    .astype({"patient": "category", "spatialleiden": "category", "sample": "category"})
)

for t, t_df in labels.groupby("tool", observed=True):
    j = 0
    for p, p_df in t_df.groupby("patient", observed=True):
        for s, df in p_df.groupby("sample", observed=True):
            ax = axs[i, j]
            sns.scatterplot(
                data=df,
                x="x",
                y="y",
                hue="spatialleiden",
                ax=ax,
                legend=False,
                **scatter_kwargs,
            )
            remove_tick_and_label(ax)
            ax.set(aspect=1)
            if j == 0:
                ax.set(ylabel=t)
            j += 1
    i += 1


fig.tight_layout()

In [ ]:
def get_metric(func, metric_name: str):
    metrics = list()

    for file in libd_dlpfc_results.glob("*/*/*.parquet"):
        tool = file.parents[1].name
        batch = file.parent.name
        name = file.stem
        labels = pd.read_parquet(file).astype({"spatialleiden": "category"})
        grouping = [x for x in ["subject", "sample_id"] if x in labels.columns]

        for g, df in labels.groupby(grouping, observed=True):
            sample = g[1] if len(g) > 1 else g[0]
            patient = g[0] if len(g) > 1 else name
            metric = func(df["label"].cat.codes, df["spatialleiden"].cat.codes)
            metrics.append(
                {
                    metric_name: metric,
                    "sample": sample,
                    "patient": patient,
                    "clustering": batch,
                    "tool": tool,
                }
            )

    for file in libd_dlpfc_results.glob("*/*.parquet"):
        patient, sample = file.stem.split("_")
        labels = pd.read_parquet(file, columns=["label", "spatialleiden"]).astype(
            "category"
        )
        metric = func(labels["label"].cat.codes, labels["spatialleiden"].cat.codes)
        metrics.append(
            {
                metric_name: metric,
                "sample": sample,
                "clustering": "sample",
                "tool": "no_batch_integration",
                "patient": patient,
            }
        )

    metrics = pd.DataFrame(metrics).assign(
        tool=lambda df: df["tool"].replace(integration),
        clustering=lambda df: df["clustering"]
        .astype("category")
        .cat.reorder_categories(["sample", "patient", "all"]),
    )

    return metrics

In [ ]:
ari_df = get_metric(adjusted_rand_score, "ARI")
nmi_df = get_metric(normalized_mutual_info_score, "NMI")

In [ ]:
def custom_swarm(data, **kwargs):
    order = [i for i in integration.values() if i in data[kwargs["x"]].unique()]
    sns.swarmplot(data=data, order=order, **kwargs)

In [ ]:
g = sns.FacetGrid(
    data=ari_df,
    sharex=False,
    col="clustering",
    gridspec_kws={"width_ratios": [1, 5, 5]},
)
g.map_dataframe(
    custom_swarm, x="tool", y="ARI", hue="patient", palette="colorblind", dodge=True
)
_ = g.add_legend(title="patient")
_ = g.set(ylim=(0, 0.8), xlabel="Batch integration")
_ = g.set_titles(col_template="{col_name}")

In [ ]:
g = sns.FacetGrid(
    data=nmi_df,
    sharex=False,
    col="clustering",
    gridspec_kws={"width_ratios": [1, 5, 5]},
)
g.map_dataframe(
    custom_swarm, x="tool", y="NMI", hue="patient", palette="colorblind", dodge=True
)
_ = g.add_legend(title="patient")
_ = g.set(ylim=(0, 0.8), xlabel="Batch integration")
_ = g.set_titles(col_template="{col_name}")